<a href="https://colab.research.google.com/github/ouchn2580201251107/statistics_homework_2026_BUCT/blob/master/%E7%BB%9F%E8%AE%A1%E5%AD%A6%E4%BD%9C%E4%B8%9A5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import pearsonr
import numpy as np

# 数据预处理
df = pd.read_csv('/content/hw5ex1.csv')

df = df[df['家庭'] != '合计']
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df['收入X'] = pd.to_numeric(df['收入X'], errors='coerce')
df['支出 Y'] = pd.to_numeric(df['支出 Y'], errors='coerce')
df = df.dropna(subset=['收入X', '支出 Y'])
df = df.rename(columns={'收入X': 'income', '支出 Y': 'expenditure'})

# (1) 试求回归方程
model = smf.ols('expenditure ~ income', data=df).fit()

b0 = model.params['Intercept']
b1 = model.params['income']

print("(1) 回归方程:")
print(f"Y = {b0:.4f} + {b1:.4f}X")
print(f"其中 Y 代表食品支出，X 代表家庭收入。\n")

# (2) 试求相关系数及拟合度，并说明两个变量之间相关关系强弱
correlation_coefficient, _ = pearsonr(df['income'], df['expenditure'])
r_squared = model.rsquared

print("(2) 相关系数及拟合度:")
print(f"相关系数: {correlation_coefficient:.4f}")
print(f"拟合度: {r_squared:.4f}")

abs_corr = abs(correlation_coefficient)
if abs_corr >= 0.8:
    strength = "非常强"
elif abs_corr >= 0.6:
    strength = "强"
elif abs_corr >= 0.4:
    strength = "中等"
elif abs_corr >= 0.2:
    strength = "弱"
else:
    strength = "非常弱或没有"

print(f"两个变量之间相关关系强度: {strength} ({'正相关' if correlation_coefficient > 0 else '负相关' if correlation_coefficient < 0 else '无相关'})\n")

# (3) 显著性水平0.05下，检验相关系数的显著性
critical_r = 0.6320

print("(3) 相关系数的显著性检验:")
print(f"显著性水平: 0.05")
print(f"样本量 (n): {len(df)}")
print(f"自由度 (df = n-2): {len(df) - 2}")
print(f"给定临界相关系数 R0.05({len(df)-2}): {critical_r:.4f}")
print(f"计算得到的|r|: {abs(correlation_coefficient):.4f}")

if abs(correlation_coefficient) > critical_r:
    print("因为 |r| > 临界相关系数 R0.05(8)，所以拒绝原假设 (H0: ρ=0)。\n结论: 在0.05的显著性水平下，相关系数显著，食品支出与家庭收入之间存在显著的线性关系。")
else:
    print("因为 |r| <= 临界相关系数 R0.05(8)，所以不拒绝原假设 (H0: ρ=0)。\n结论: 在0.05的显著性水平下，相关系数不显著，食品支出与家庭收入之间不存在显著的线性关系。")

print("\n（替代方法: 使用t检验检验斜率的显著性）")

print(model.summary().tables[1])

t_value_income = model.tvalues['income']
p_value_income = model.pvalues['income']

critical_t = 2.306

print(f"计算得到的 t 值: {t_value_income:.4f}")
print(f"给定临界 t 值 t0.025({len(df)-2}): {critical_t:.4f}")
print(f"p 值: {p_value_income:.4f}")

if abs(t_value_income) > critical_t:
    print("因为 |t| > 临界 t 值，所以拒绝原假设 (H0: β1=0)。")
    print("结论: 在0.05的显著性水平下，家庭收入对食品支出的影响显著。")
elif p_value_income < 0.05:
    print("因为 p 值 < 0.05，所以拒绝原假设 (H0: β1=0)。")
    print("结论: 在0.05的显著性水平下，家庭收入对食品支出的影响显著。")
else:
    print("因为 |t| <= 临界 t 值 (或 p 值 >= 0.05)，所以不拒绝原假设 (H0: β1=0)。")
    print("结论: 在0.05的显著性水平下，家庭收入对食品支出的影响不显著。")

print("\n")

# (4) 有一个家庭A食品收入为22，试求支出
income_a = pd.DataFrame({'income': [22]})
predicted_expenditure_a = model.predict(income_a)[0]

print("(4) 家庭A食品收入为22时的预测支出 = 22):")
print(f"当家庭收入为22时，预测食品支出为: {predicted_expenditure_a:.4f}\n")

# (5) 试以95%的概率保证程度估计A家庭的总收入的置信区间。
predictions = model.get_prediction(income_a)
summary_frame = predictions.summary_frame(alpha=0.05)

lower_bound = summary_frame['obs_ci_lower'][0]
upper_bound = summary_frame['obs_ci_upper'][0]

print("(5) 估计A家庭食品支出的95%预测区间:")
print(f"对于收入为22的家庭，其食品支出的95%预测区间为: [{lower_bound:.4f}, {upper_bound:.4f}]\n")

(1) 回归方程:
Y = 2.1727 + 0.2023X
其中 Y 代表食品支出，X 代表家庭收入。

(2) 相关系数及拟合度:
相关系数: 0.9509
拟合度: 0.9043
两个变量之间相关关系强度: 非常强 (正相关)

(3) 相关系数的显著性检验:
显著性水平: 0.05
样本量 (n): 10
自由度 (df = n-2): 8
给定临界相关系数 R0.05(8): 0.6320
计算得到的|r|: 0.9509
因为 |r| > 临界相关系数 R0.05(8)，所以拒绝原假设 (H0: ρ=0)。
结论: 在0.05的显著性水平下，相关系数显著，食品支出与家庭收入之间存在显著的线性关系。

（替代方法: 使用t检验检验斜率的显著性）
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      2.1727      0.720      3.017      0.017       0.512       3.833
income         0.2023      0.023      8.692      0.000       0.149       0.256
计算得到的 t 值: 8.6925
给定临界 t 值 t0.025(8): 2.3060
p 值: 0.0000
因为 |t| > 临界 t 值，所以拒绝原假设 (H0: β1=0)。
结论: 在0.05的显著性水平下，家庭收入对食品支出的影响显著。


(4) 家庭A食品收入为22时的预测支出 = 22):
当家庭收入为22时，预测食品支出为: 6.6232

(5) 估计A家庭食品支出的95%预测区间:
对于收入为22的家庭，其食品支出的95%预测区间为: [4.8076, 8.4389]



In [10]:
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import pearsonr
import numpy as np

df = pd.read_csv('/content/hw5ex2.csv')

df = df.iloc[1:].copy()
df.columns = ['林场编号', '年木材剩余Y', '年木材采伐X', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5']
df = df[['年木材剩余Y', '年木材采伐X']].copy()
df['年木材剩余Y'] = pd.to_numeric(df['年木材剩余Y'], errors='coerce')
df['年木材采伐X'] = pd.to_numeric(df['年木材采伐X'], errors='coerce')
df = df.dropna()
df = df.rename(columns={'年木材采伐X': 'logging_volume', '年木材剩余Y': 'timber_remainder'})

# (1) 试求一元回归方程
model = smf.ols('timber_remainder ~ logging_volume', data=df).fit()

b0 = model.params['Intercept']
b1 = model.params['logging_volume']

print("(1) 回归方程:")
print(f"Y = {b0:.2f} + {b1:.2f}X")
print(f"其中 Y 代表年木材剩余量，X 代表年木材采伐量。\n")

# (2) 试求相关系数及拟合度，并说明两个变量之间相关关系强弱
correlation_coefficient, _ = pearsonr(df['logging_volume'], df['timber_remainder'])
r_squared = model.rsquared

print("(2) 相关系数及拟合度:")
print(f"相关系数: {correlation_coefficient:.4f}")
print(f"拟合度: {r_squared:.4f}")

abs_corr = abs(correlation_coefficient)
if abs_corr >= 0.8:
    strength = "非常强"
elif abs_corr >= 0.6:
    strength = "强"
elif abs_corr >= 0.4:
    strength = "中等"
elif abs_corr >= 0.2:
    strength = "弱"
else:
    strength = "非常弱或没有"

print(f"两个变量之间相关关系强度: {strength} ({'正相关' if correlation_coefficient > 0 else '负相关' if correlation_coefficient < 0 else '无相关'})\n")

# (3) 显著性水平0.05下，检验相关系数的显著性
n = len(df)
df_corr = n - 2
critical_r = 0.6320

print("(3) 相关系数的显著性检验:")
print(f"显著性水平: 0.05")
print(f"样本量 (n): {n}")
print(f"自由度 (df = n-2): {df_corr}")
print(f"给定临界相关系数 R0.05({df_corr}): {critical_r:.4f}")
print(f"计算得到的|r|: {abs(correlation_coefficient):.4f}")

if abs(correlation_coefficient) > critical_r:
    print("因为 |r| > 临界相关系数 R0.05(8)，所以拒绝原假设 (H0: ρ=0).")
    print("结论: 在0.05的显著性水平下，相关系数显著，年木材剩余量与年木材采伐量之间存在显著的线性关系。\n")
elif abs(correlation_coefficient) <= critical_r:
    print("因为 |r| <= 临界相关系数 R0.05(8)，所以不拒绝原假设 (H0: ρ=0).")
    print("结论: 在0.05的显著性水平下，相关系数不显著，年木材剩余量与年木材采伐量之间不存在显著的线性关系。\n")

print("（替代方法: 使用t检验检验斜率的显著性）")
print(model.summary().tables[1])

t_value_logging = model.tvalues['logging_volume']
p_value_logging = model.pvalues['logging_volume']
critical_t = 2.306

print(f"计算得到的 t 值: {t_value_logging:.2f}")
print(f"给定临界 t 值 t0.025({df_corr}): {critical_t:.3f}")
print(f"p 值: {p_value_logging:.2f}")

if abs(t_value_logging) > critical_t:
    print("因为 |t| > 临界 t 值，所以拒绝原假设 (H0: β1=0).")
    print("结论: 在0.05的显著性水平下，年木材采伐量对年木材剩余量的影响显著。\n")
elif p_value_logging < 0.05:
    print("因为 p 值 < 0.05，所以拒绝原假设 (H0: β1=0).")
    print("结论: 在0.05的显著性水平下，年木材采伐量对年木材剩余量的影响显著。\n")
else:
    print("因为 |t| <= 临界 t 值 (或 p 值 >= 0.05)，所以不拒绝原假设 (H0: β1=0).")
    print("结论: 在0.05的显著性水平下，年木材采伐量对年木材剩余量的影响不显著。\n")

# (4) 2018年计划采伐木材20万立方米，试求年木材剩余量
logging_2018 = pd.DataFrame({'logging_volume': [20]})
predicted_remainder_2018 = model.predict(logging_2018)[0]

print("(4) 2018年计划采伐木材20万立方米时的年木材剩余量:")
print(f"当采伐量为20万立方米时，预测年木材剩余量为: {predicted_remainder_2018:.2f}\n")

# (5) 试以95%的概率保证程度估计林区木材剩余量的置信区间。
predictions = model.get_prediction(logging_2018)
summary_frame = predictions.summary_frame(alpha=0.05)

lower_bound = summary_frame['obs_ci_lower'][0]
upper_bound = summary_frame['obs_ci_upper'][0]

print("(5) 估计林区木材剩余量的95%预测区间 (采伐量为20万立方米时):")
print(f"对于采伐量为20万立方米的林场，其年木材剩余量的95%预测区间为: [{lower_bound:.2f}, {upper_bound:.2f}]")

(1) 回归方程:
Y = -0.94 + 0.42X
其中 Y 代表年木材剩余量，X 代表年木材采伐量。

(2) 相关系数及拟合度:
相关系数: 0.9500
拟合度: 0.9024
两个变量之间相关关系强度: 非常强 (正相关)

(3) 相关系数的显著性检验:
显著性水平: 0.05
样本量 (n): 10
自由度 (df = n-2): 8
给定临界相关系数 R0.05(8): 0.6320
计算得到的|r|: 0.9500
因为 |r| > 临界相关系数 R0.05(8)，所以拒绝原假设 (H0: ρ=0).
结论: 在0.05的显著性水平下，相关系数显著，年木材剩余量与年木材采伐量之间存在显著的线性关系。

（替代方法: 使用t检验检验斜率的显著性）
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -0.9420      1.930     -0.488      0.639      -5.393       3.509
logging_volume     0.4176      0.049      8.601      0.000       0.306       0.530
计算得到的 t 值: 8.60
给定临界 t 值 t0.025(8): 2.306
p 值: 0.00
因为 |t| > 临界 t 值，所以拒绝原假设 (H0: β1=0).
结论: 在0.05的显著性水平下，年木材采伐量对年木材剩余量的影响显著。

(4) 2018年计划采伐木材20万立方米时的年木材剩余量:
当采伐量为20万立方米时，预测年木材剩余量为: 7.41

(5) 估计林区木材剩余量的95%预测区间 (采伐量为20万立方米时):
对于采伐量为20万立方米的林场，其年木材剩余量的95%预测区间为: [1.27, 13.55]
